In [ ]:
### EXP 1：one stage classifier 30 epochs

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = '/content/MA'
output_dir = base_path 

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}

class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx])  
        if mel.ndim == 2: 
            mel = mel.unsqueeze(0) 
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7):
        super(ResNet50Classifier, self).__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.base_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.base_model.fc = nn.Linear(self.base_model.fc.in_features, num_classes)

    def forward(self, x):
        return self.base_model(x)

train_set_path = os.path.join(base_path, "/content/MA/train_set.xlsx")
test_set_path = os.path.join(base_path, "/content/MA/test_set.xlsx")
train_df = pd.read_excel(train_set_path)
test_df = pd.read_excel(test_set_path)

def get_file_paths_and_labels(df, base_dir):
    files, labels = [], []
    for _, row in df.iterrows():
        relative_path = row['Full_Path'] 
        pathology = row['Pathology']    
        full_path = os.path.join(base_dir, relative_path.replace('\\', '/'))  
        if os.path.exists(full_path) and pathology in classes:
            files.append(full_path)
            labels.append(classes[pathology])
        else:
            print(f"Warning: File {full_path} does not exist or invalid pathology.")
    return files, labels

train_files, train_labels = get_file_paths_and_labels(train_df, base_path)
test_files, test_labels = get_file_paths_and_labels(test_df, base_path)

test_dataset = MelDataset(test_files, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"))
    plt.close()

def save_metrics(metrics, fold):
    metrics_df = pd.DataFrame(metrics, columns=["Epoch", "MCC"])
    metrics_df.to_csv(os.path.join(output_dir, f"validation_metrics_fold_{fold}.csv"), index=False)

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes.keys(), yticklabels=classes.keys())
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"))
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.4f},{precision:.4f},{recall:.4f},{f1:.4f}\n")

cv_results = []

best_params = None
best_model = None
best_val_mcc = -float('inf')  

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset([train_files[i] for i in train_idx], [train_labels[i] for i in train_idx])
        val_dataset = MelDataset([train_files[i] for i in val_idx], [train_labels[i] for i in val_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        for epoch in range(30):
            model.train()
            total_train_loss = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

            train_loss = total_train_loss / len(train_loader)
            val_loss = total_val_loss / len(val_loader)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        mcc = matthews_corrcoef(y_true, y_pred)
        fold_metrics.append(mcc)
        print(f"Fold {fold} | MCC: {mcc:.4f}")
        fold += 1

    avg_mcc = np.mean(fold_metrics)
    std_mcc = np.std(fold_metrics)
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)
        best_model = model

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)


print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")
print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)
final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(30):
    final_model.train()
    total_train_loss = 0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(final_train_loader)
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final")


In [ ]:
### EXP 1：add hidden layers to the baseline for epoch 30

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = '/content/MA'
output_dir = base_path  

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}
class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx])  
        mel = torch.tensor(mel, dtype=torch.float32)
        if mel.ndim == 2:  
            mel = mel.unsqueeze(0)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super(ResNet50Classifier, self).__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        self.base_model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        in_feat = self.base_model.fc.in_features 

        self.base_model.fc = nn.Sequential(
            nn.Linear(in_feat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)


train_set_path = os.path.join(base_path, "/content/MA/train_set.xlsx")
test_set_path = os.path.join(base_path, "/content/MA/test_set.xlsx")
train_df = pd.read_excel(train_set_path)
test_df = pd.read_excel(test_set_path)

def get_file_paths_and_labels(df, base_dir):
    files, labels = [], []
    for _, row in df.iterrows():
        relative_path = row['Full_Path']  
        pathology = row['Pathology']     
        full_path = os.path.join(base_dir, relative_path.replace('\\', '/'))  
        if os.path.exists(full_path) and pathology in classes:
            files.append(full_path)
            labels.append(classes[pathology])
        else:
            print(f"Warning: File {full_path} does not exist or invalid pathology.")
    return files, labels

train_files, train_labels = get_file_paths_and_labels(train_df, base_path)
test_files, test_labels = get_file_paths_and_labels(test_df, base_path)

test_dataset = MelDataset(test_files, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"))
    plt.close()

def save_metrics(metrics, fold):
    metrics_df = pd.DataFrame(metrics, columns=["Epoch", "MCC"])
    metrics_df.to_csv(os.path.join(output_dir, f"validation_metrics_fold_{fold}.csv"), index=False)

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes.keys(), yticklabels=classes.keys())
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"))
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.4f},{precision:.4f},{recall:.4f},{f1:.4f}\n")

cv_results = []

best_params = None
best_model = None
best_val_mcc = -float('inf')  

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset([train_files[i] for i in train_idx], [train_labels[i] for i in train_idx])
        val_dataset = MelDataset([train_files[i] for i in val_idx], [train_labels[i] for i in val_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        for epoch in range(30):
            model.train()
            total_train_loss = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

            train_loss = total_train_loss / len(train_loader)
            val_loss = total_val_loss / len(val_loader)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        mcc = matthews_corrcoef(y_true, y_pred)
        fold_metrics.append(mcc)
        print(f"Fold {fold} | MCC: {mcc:.4f}")
        fold += 1

    avg_mcc = np.mean(fold_metrics)
    std_mcc = np.std(fold_metrics)
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)
        best_model = model

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)


print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")
print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)
final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(30):
    final_model.train()
    total_train_loss = 0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(final_train_loader)
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final")


Using device: cuda
Testing parameter combination: batch_size=32, lr=0.001
Starting Fold 1 with batch_size=32, lr=0.001
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 244MB/s]


Epoch 1: Train Loss: 0.8544, Validation Loss: 0.8038
Epoch 2: Train Loss: 0.7342, Validation Loss: 0.6729
Epoch 3: Train Loss: 0.6622, Validation Loss: 0.6501
Epoch 4: Train Loss: 0.6274, Validation Loss: 0.5629
Epoch 5: Train Loss: 0.5961, Validation Loss: 0.5685
Epoch 6: Train Loss: 0.5666, Validation Loss: 0.6043
Epoch 7: Train Loss: 0.5056, Validation Loss: 0.4724
Epoch 8: Train Loss: 0.4817, Validation Loss: 0.5515
Epoch 9: Train Loss: 0.4431, Validation Loss: 0.4495
Epoch 10: Train Loss: 0.4224, Validation Loss: 0.5336
Epoch 11: Train Loss: 0.3911, Validation Loss: 0.5253
Epoch 12: Train Loss: 0.3584, Validation Loss: 0.3837
Epoch 13: Train Loss: 0.3231, Validation Loss: 0.5067
Epoch 14: Train Loss: 0.3152, Validation Loss: 0.3637
Epoch 15: Train Loss: 0.2880, Validation Loss: 0.5667
Epoch 16: Train Loss: 0.2682, Validation Loss: 0.3958
Epoch 17: Train Loss: 0.2472, Validation Loss: 0.3734
Epoch 18: Train Loss: 0.2205, Validation Loss: 0.3532
Epoch 19: Train Loss: 0.2491, Validat

In [ ]:
### EXP 1：add hidden layers to the baseline for epoch 20

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = '/content/MA'
output_dir = base_path  

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}
class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx])  
        mel = torch.tensor(mel, dtype=torch.float32)
        if mel.ndim == 2:  
            mel = mel.unsqueeze(0)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super(ResNet50Classifier, self).__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        self.base_model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        in_feat = self.base_model.fc.in_features 

        self.base_model.fc = nn.Sequential(
            nn.Linear(in_feat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)


train_set_path = os.path.join(base_path, "/content/MA/train_set.xlsx")
test_set_path = os.path.join(base_path, "/content/MA/test_set.xlsx")
train_df = pd.read_excel(train_set_path)
test_df = pd.read_excel(test_set_path)

def get_file_paths_and_labels(df, base_dir):
    files, labels = [], []
    for _, row in df.iterrows():
        relative_path = row['Full_Path'] 
        pathology = row['Pathology']     
        full_path = os.path.join(base_dir, relative_path.replace('\\', '/'))  
        if os.path.exists(full_path) and pathology in classes:
            files.append(full_path)
            labels.append(classes[pathology])
        else:
            print(f"Warning: File {full_path} does not exist or invalid pathology.")
    return files, labels

train_files, train_labels = get_file_paths_and_labels(train_df, base_path)
test_files, test_labels = get_file_paths_and_labels(test_df, base_path)

test_dataset = MelDataset(test_files, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"))
    plt.close()

def save_metrics(metrics, fold):
    metrics_df = pd.DataFrame(metrics, columns=["Epoch", "MCC"])
    metrics_df.to_csv(os.path.join(output_dir, f"validation_metrics_fold_{fold}.csv"), index=False)

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes.keys(), yticklabels=classes.keys())
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"))
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.4f},{precision:.4f},{recall:.4f},{f1:.4f}\n")

cv_results = []

best_params = None
best_model = None
best_val_mcc = -float('inf')  

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset([train_files[i] for i in train_idx], [train_labels[i] for i in train_idx])
        val_dataset = MelDataset([train_files[i] for i in val_idx], [train_labels[i] for i in val_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        for epoch in range(20):
            model.train()
            total_train_loss = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

            train_loss = total_train_loss / len(train_loader)
            val_loss = total_val_loss / len(val_loader)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        mcc = matthews_corrcoef(y_true, y_pred)
        fold_metrics.append(mcc)
        print(f"Fold {fold} | MCC: {mcc:.4f}")
        fold += 1

    avg_mcc = np.mean(fold_metrics)
    std_mcc = np.std(fold_metrics)
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)
        best_model = model

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)


print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")
print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)
final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(20):
    final_model.train()
    total_train_loss = 0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(final_train_loader)
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final")


Using device: cuda
Testing parameter combination: batch_size=32, lr=0.001
Starting Fold 1 with batch_size=32, lr=0.001
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 216MB/s]


Epoch 1: Train Loss: 0.8989, Validation Loss: 0.8128
Epoch 2: Train Loss: 0.8127, Validation Loss: 0.7735
Epoch 3: Train Loss: 0.7288, Validation Loss: 0.7000
Epoch 4: Train Loss: 0.6422, Validation Loss: 0.6193
Epoch 5: Train Loss: 0.5840, Validation Loss: 0.6570
Epoch 6: Train Loss: 0.5461, Validation Loss: 0.7307
Epoch 7: Train Loss: 0.5026, Validation Loss: 0.5525
Epoch 8: Train Loss: 0.4621, Validation Loss: 0.5956
Epoch 9: Train Loss: 0.4293, Validation Loss: 0.4316
Epoch 10: Train Loss: 0.3871, Validation Loss: 0.5415
Epoch 11: Train Loss: 0.3767, Validation Loss: 0.4997
Epoch 12: Train Loss: 0.3470, Validation Loss: 0.3689
Epoch 13: Train Loss: 0.3022, Validation Loss: 0.4254
Epoch 14: Train Loss: 0.2789, Validation Loss: 0.3523
Epoch 15: Train Loss: 0.2542, Validation Loss: 0.6603
Epoch 16: Train Loss: 0.2424, Validation Loss: 0.3876
Epoch 17: Train Loss: 0.2238, Validation Loss: 0.3513
Epoch 18: Train Loss: 0.2092, Validation Loss: 0.4561
Epoch 19: Train Loss: 0.2099, Validat

In [ ]:
### EXP 1：add hidden layers to the baseline for epoch 10

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = '/content/MA'
output_dir = base_path  

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}

class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx])  
        mel = torch.tensor(mel, dtype=torch.float32)
        if mel.ndim == 2:  
            mel = mel.unsqueeze(0)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super(ResNet50Classifier, self).__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        self.base_model.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        in_feat = self.base_model.fc.in_features  

        self.base_model.fc = nn.Sequential(
            nn.Linear(in_feat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)


train_set_path = os.path.join(base_path, "/content/MA/train_set.xlsx")
test_set_path = os.path.join(base_path, "/content/MA/test_set.xlsx")
train_df = pd.read_excel(train_set_path)
test_df = pd.read_excel(test_set_path)

def get_file_paths_and_labels(df, base_dir):
    files, labels = [], []
    for _, row in df.iterrows():
        relative_path = row['Full_Path'] 
        pathology = row['Pathology']    
        full_path = os.path.join(base_dir, relative_path.replace('\\', '/'))  
        if os.path.exists(full_path) and pathology in classes:
            files.append(full_path)
            labels.append(classes[pathology])
        else:
            print(f"Warning: File {full_path} does not exist or invalid pathology.")
    return files, labels

train_files, train_labels = get_file_paths_and_labels(train_df, base_path)
test_files, test_labels = get_file_paths_and_labels(test_df, base_path)

test_dataset = MelDataset(test_files, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"))
    plt.close()

def save_metrics(metrics, fold):
    metrics_df = pd.DataFrame(metrics, columns=["Epoch", "MCC"])
    metrics_df.to_csv(os.path.join(output_dir, f"validation_metrics_fold_{fold}.csv"), index=False)

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes.keys(), yticklabels=classes.keys())
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"))
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.4f},{precision:.4f},{recall:.4f},{f1:.4f}\n")

cv_results = []

best_params = None
best_model = None
best_val_mcc = -float('inf')  
for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset([train_files[i] for i in train_idx], [train_labels[i] for i in train_idx])
        val_dataset = MelDataset([train_files[i] for i in val_idx], [train_labels[i] for i in val_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        for epoch in range(10):
            model.train()
            total_train_loss = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

            train_loss = total_train_loss / len(train_loader)
            val_loss = total_val_loss / len(val_loader)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        mcc = matthews_corrcoef(y_true, y_pred)
        fold_metrics.append(mcc)
        print(f"Fold {fold} | MCC: {mcc:.4f}")
        fold += 1

    avg_mcc = np.mean(fold_metrics)
    std_mcc = np.std(fold_metrics)
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)
        best_model = model

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)


print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")
print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)
final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(10):
    final_model.train()
    total_train_loss = 0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(final_train_loader)
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final")


Using device: cuda
Testing parameter combination: batch_size=32, lr=0.001
Starting Fold 1 with batch_size=32, lr=0.001
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


Epoch 1: Train Loss: 0.8602, Validation Loss: 0.7557
Epoch 2: Train Loss: 0.7402, Validation Loss: 0.7503
Epoch 3: Train Loss: 0.6916, Validation Loss: 0.6377
Epoch 4: Train Loss: 0.6308, Validation Loss: 0.5693
Epoch 5: Train Loss: 0.5827, Validation Loss: 0.5865
Epoch 6: Train Loss: 0.5353, Validation Loss: 0.5437
Epoch 7: Train Loss: 0.4926, Validation Loss: 0.4945
Epoch 8: Train Loss: 0.4549, Validation Loss: 0.5124
Epoch 9: Train Loss: 0.4343, Validation Loss: 0.5733
Epoch 10: Train Loss: 0.4015, Validation Loss: 0.6038
Fold 1 | MCC: 0.5999
Starting Fold 2 with batch_size=32, lr=0.001
Epoch 1: Train Loss: 0.8730, Validation Loss: 0.8498
Epoch 2: Train Loss: 0.7316, Validation Loss: 0.9774
Epoch 3: Train Loss: 0.6495, Validation Loss: 0.8279
Epoch 4: Train Loss: 0.6057, Validation Loss: 0.6378
Epoch 5: Train Loss: 0.5607, Validation Loss: 0.5257
Epoch 6: Train Loss: 0.5129, Validation Loss: 0.5194
Epoch 7: Train Loss: 0.4825, Validation Loss: 0.4845
Epoch 8: Train Loss: 0.4477, Val